In [1]:
import torch
import os, sys

import polars as pl


sys.path.insert(0, os.path.abspath(".."))

MAC_DIR = '/Users/igwanhyeong/PycharmProjects/paper_research/'
WINDOW_DIR = 'C:/Users/USER/PycharmProjects/paper_research/'

if sys.platform == 'win32':
    DIR = WINDOW_DIR
    print(torch.cuda.is_available())
    print(torch.cuda.device_count())
    print(torch.version.cuda)
    print(torch.__version__)
    print(torch.cuda.get_device_name(0))
    print(torch.__version__)
else:
    DIR = MAC_DIR

In [2]:
from models.RMTPPs.TitanTPP import TitanTPP

from models.RMTPPs.config import RMTPPConfig
from models.Titan import TitanConfig

device = torch.device('cuda' if torch.cuda.is_available() else 'mps')
num_marks = 8
seq_len = 16
batch_size = 4

rmtpp_cfg = RMTPPConfig(
    num_marks = num_marks,
    mark_emb_dim=16,
    rnn_hidden_dim=128,
    rnn_type='gru',
    dropout=0.0,
    eps=1e-8,
    w_min=1e-3,
    exp_clamp=50.0,
)

titan_cfg = TitanConfig(
    lookback=seq_len,
    horizon=1,
    d_model=64,
    n_layers=2,
    n_heads=4,
    d_ff=128,
    dropout=0.0,
    contextual_mem_size=8,
    persistent_mem_size=8,
    use_context_update=False,
    use_pos_emb=True,
    max_len=128,
    use_lmm=False,
    mem_size=32,
    mem_topk=4,
    use_causal=True,
    use_revin=True,
)

model = TitanTPP(rmtpp_cfg, titan_cfg).to(device)
model.eval()

print(model)
print('w_pos =', model._w_pos().item())

TitanTPP(
  (emb): Embedding(8, 16)
  (encoder): MemoryEncoder(
    (input_proj): Linear(in_features=17, out_features=64, bias=True)
    (layers): ModuleList(
      (0-1): 2 x TitanBackbone(
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (attn): MemoryAttention(
          (qkv): Linear(in_features=64, out_features=192, bias=True)
          (out_proj): Linear(in_features=64, out_features=64, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
          (_ctx_buf): ContextualMemoryBuffer()
        )
        (drop1): Dropout(p=0.0, inplace=False)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (ff): Sequential(
          (0): Linear(in_features=64, out_features=128, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.0, inplace=False)
          (3): Linear(in_features=128, out_features=64, bias=True)
        )
        (drop2): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (mark_head): Linear(

### Make dummy batch

In [3]:
def make_dummy_batch(batch_size=4, seq_len=16, num_marks=8, device = 'cpu'):
    marks = torch.randint(
        low=0,
        high=num_marks,
        size=(batch_size, seq_len),
        device=device,
    )

    # inter-event time: 0 이상
    dts = torch.randint(
        low=0,
        high=10,
        size=(batch_size, seq_len),
        device=device,
    ).float()

    dts[:, 0] = 0.0

    mask = torch.ones((batch_size, seq_len), dtype=torch.bool, device=device)

    return marks, dts, mask

marks, dts, mask = make_dummy_batch(
    batch_size=batch_size,
    seq_len=seq_len,
    num_marks=num_marks,
    device=device
)

print('marks.shape:', marks.shape)
print('dts.shape:', dts.shape)
print('mask.shape:', mask.shape)
print('marks[0]:', marks[0])
print('dts[0]:', dts[0])



marks.shape: torch.Size([4, 16])
dts.shape: torch.Size([4, 16])
mask.shape: torch.Size([4, 16])
marks[0]: tensor([5, 4, 1, 2, 2, 1, 7, 4, 0, 4, 3, 2, 6, 1, 7, 7], device='mps:0')
dts[0]: tensor([0., 9., 3., 2., 5., 3., 3., 2., 0., 3., 2., 7., 3., 9., 1., 8.],
       device='mps:0')


### forward/nll smoke test

In [4]:
model.eval()
with torch.no_grad():
    h = model.forward(marks, dts)
    out = model.nll(marks, dts, mask)

print('h.shape:', h.shape)
print('nll keys:', out.keys())

for k, v in out.items():
    if torch.is_tensor(v):
        print(f"{k}: shape={tuple(v.shape)}, value = {v.detach().cpu()}")
    else:
        print(f"{k}: {v}")

assert h.shape == (batch_size, seq_len, titan_cfg.d_model)
assert torch.isfinite(h).all(), 'forward output contains NaN/Inf'

for k in ['nll', 'nll_marker', 'nll_time']:
    assert torch.isfinite(out[k]).all(), f"{k} contains NaN/Inf"

print('forward / nll smoke test passed')

h.shape: torch.Size([4, 16, 64])
nll keys: dict_keys(['nll', 'nll_marker', 'nll_time', 'steps'])
nll: shape=(), value = 6.8197479248046875
nll_marker: shape=(), value = 2.092881917953491
nll_time: shape=(), value = 4.726865768432617
steps: shape=(), value = 60
forward / nll smoke test passed


### backward / gradient flow test

In [5]:
model.train()
model.zero_grad(set_to_none=True)
out = model.nll(marks, dts, mask)
loss = out['nll']

print('loss before backward:', loss.item())

loss.backward()

grad_stats = []
for name, p in model.named_parameters():
    if p.grad is not None:
        grad_norm = p.grad.norm().item()
        grad_finite = torch.isfinite(p.grad).all().item()
        grad_stats.append((name, grad_norm, grad_finite))


print("=== Gradient Stats ===")
for name, grad_norm, grad_finite in grad_stats[:20]:
    print(f"{name:40s} norm={grad_norm:.6f}, finite={grad_finite}")

assert len(grad_stats) > 0, "No gradients found"
assert all(gf for _, _, gf in grad_stats), "Some gradients contain NaN/Inf"

print('backward / gradient flow test passed')

loss before backward: 6.8197479248046875
=== Gradient Stats ===
b_t                                      norm=3.870660, finite=True
w_raw                                    norm=0.599799, finite=True
emb.weight                               norm=0.085926, finite=True
encoder.pos_emb                          norm=0.083735, finite=True
encoder.input_proj.weight                norm=0.641246, finite=True
encoder.input_proj.bias                  norm=0.278817, finite=True
encoder.layers.0.norm1.weight            norm=0.021329, finite=True
encoder.layers.0.norm1.bias              norm=0.042951, finite=True
encoder.layers.0.attn.persistent_mem     norm=0.025290, finite=True
encoder.layers.0.attn.qkv.weight         norm=0.286482, finite=True
encoder.layers.0.attn.qkv.bias           norm=0.064536, finite=True
encoder.layers.0.attn.out_proj.weight    norm=0.375076, finite=True
encoder.layers.0.attn.out_proj.bias      norm=0.265116, finite=True
encoder.layers.0.norm2.weight            norm=0.0304

### Optimizer 1 step test

In [6]:
model.train()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# update 전 파라미터 복사
before = {name: p.detach().clone() for name, p in model.named_parameters()}

optimizer.zero_grad(set_to_none=True)
out = model.nll(marks, dts, mask)
loss = out["nll"]
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()

changed = []
for name, p in model.named_parameters():
    if not torch.equal(before[name], p.detach()):
        changed.append(name)

print("changed params:", len(changed))
print(changed[:20])

assert len(changed) > 0, "No parameter was updated"
print("optimizer step test passed")

changed params: 35
['b_t', 'w_raw', 'emb.weight', 'encoder.pos_emb', 'encoder.input_proj.weight', 'encoder.input_proj.bias', 'encoder.layers.0.norm1.weight', 'encoder.layers.0.norm1.bias', 'encoder.layers.0.attn.persistent_mem', 'encoder.layers.0.attn.qkv.weight', 'encoder.layers.0.attn.qkv.bias', 'encoder.layers.0.attn.out_proj.weight', 'encoder.layers.0.attn.out_proj.bias', 'encoder.layers.0.norm2.weight', 'encoder.layers.0.norm2.bias', 'encoder.layers.0.ff.0.weight', 'encoder.layers.0.ff.0.bias', 'encoder.layers.0.ff.3.weight', 'encoder.layers.0.ff.3.bias', 'encoder.layers.1.norm1.weight']
optimizer step test passed


### Tiny batch overfit test

In [7]:
# 작은 배치 1개 고정
marks_small, dts_small, mask_small = make_dummy_batch(
    batch_size=2,
    seq_len=12,
    num_marks=num_marks,
    device=device
)

model = TitanTPP(rmtpp_cfg, titan_cfg).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

loss_history = []

model.train()
for step in range(300):
    optimizer.zero_grad(set_to_none=True)
    out = model.nll(marks_small, dts_small, mask_small)
    loss = out["nll"]
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    loss_history.append(loss.item())

    if step % 50 == 0:
        print(
            f"step={step:03d} | "
            f"nll={out['nll'].item():.6f} | "
            f"marker={out['nll_marker'].item():.6f} | "
            f"time={out['nll_time'].item():.6f}"
        )

print("first loss:", loss_history[0])
print("last loss :", loss_history[-1])

assert loss_history[-1] < loss_history[0], "Loss did not decrease"
print("tiny batch overfit test passed")

step=000 | nll=6.763271 | marker=2.092998 | time=4.670273
step=050 | nll=2.029056 | marker=0.075091 | time=1.953965
step=100 | nll=-2.370980 | marker=0.019085 | time=-2.390064
step=150 | nll=-2.424893 | marker=0.002799 | time=-2.427692
step=200 | nll=-2.416386 | marker=0.001054 | time=-2.417440
step=250 | nll=-2.462783 | marker=0.000474 | time=-2.463257
first loss: 6.763270854949951
last loss : -2.473560333251953
tiny batch overfit test passed


### sample_next_dt sanity check

In [8]:
model.eval()

with torch.no_grad():
    h = model.forward(marks, dts)        # [B, L, D]
    h_j = h[:, -1, :]                    # [B, D]
    sampled_dt = model.sample_next_dt(h_j)

print("sampled_dt.shape:", sampled_dt.shape)
print("sampled_dt:", sampled_dt)

assert torch.isfinite(sampled_dt).all(), "sample_next_dt contains NaN/Inf"
assert (sampled_dt >= 0).all(), "sampled_dt has negative values"

print("sample_next_dt sanity test passed")

sampled_dt.shape: torch.Size([4])
sampled_dt: tensor([1.1465e+01, 5.7881e-01, 4.4537e+00, 3.5450e-06], device='mps:0')
sample_next_dt sanity test passed


### Causal masking sanity check

In [9]:
# LMM=False 상태의 model 사용
model = TitanTPP(rmtpp_cfg, titan_cfg).to(device)
model.eval()

marks_a, dts_a, _ = make_dummy_batch(
    batch_size=1,
    seq_len=10,
    num_marks=num_marks,
    device=device
)

marks_b = marks_a.clone()
dts_b = dts_a.clone()

# 마지막 토큰만 크게 바꿔보기
marks_b[0, -1] = (marks_b[0, -1] + 3) % num_marks
dts_b[0, -1] = dts_b[0, -1] + 50.0

with torch.no_grad():
    h_a = model.forward(marks_a, dts_a)   # [1, L, D]
    h_b = model.forward(marks_b, dts_b)

# 마지막 위치를 제외한 앞부분 비교
prefix_diff = (h_a[:, :-1, :] - h_b[:, :-1, :]).abs().max().item()
last_diff = (h_a[:, -1, :] - h_b[:, -1, :]).abs().max().item()

print("prefix max diff:", prefix_diff)
print("last   max diff:", last_diff)

# causal이면 앞부분은 거의 같아야 함
assert prefix_diff < 1e-5, f"Causality broken: prefix changed by {prefix_diff}"
print("causal mask sanity test passed")

prefix max diff: 0.0
last   max diff: 2.6990082263946533
causal mask sanity test passed
